**Eye-Tracking & Neurocognitive Intelligence | Eye-Movement Modelling & Advanced AI | Presented by: Reza Saadatyar (2026)**

**01 | GazeBase Source, Target and Participant-Disjoint Protocol**

![Study 2 source selection, data representations and independent-person partitions](../../images/01_data_and_experimental_protocol.svg)

**Objective:** Build an independent supervised task-identity study from original GazeBase CSVs. The target is FXS, HSS, RAN or TEX; it is not a cognitive, clinical or comprehension label. Preserve source record and participant identities.

**Source and independence:** GazeBase v2.0, Round 1 / Session 1. Reuse original source CSVs under the common repository data directory, **not** Study 1's feature tables, processed CSVs or outputs. This notebook creates Study 2-owned data, features, windows and partitions.

**Protocol:** Start with **10 complete participants in pilot mode**. The single MAX_PARTICIPANTS parameter may later be changed to 20 or 30 only after the corresponding original source files are available. Every change requires rerunning this notebook and all downstream notebooks. For N≥20, the highest previously unseen source IDs are reserved for the main Test; the 10-participant pilot is exploratory, not definitive unseen-person validation.

**Leakage protection:** Split by participant **before any train-derived normalisation**, and never feed record IDs, task labels, subject IDs or recording duration to the models. Window labels inherit recording Task, but sliding windows do not create independent subjects. No model has been trained or evaluated in this unexecuted notebook.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_movement_modelling")

# ----------------------- 2. Read-Only Source Adapter -------------------------
import sys
source_module = str(root / "eye_tracking_foundations" / "code" / "src")
if source_module not in sys.path:
    sys.path.insert(0, source_module)

# ------------------------ 3. Scientific Data Libraries -----------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# --------------------------- 4. Study 2 Utilities ----------------------------
from gaze_source import discover_records
from s2_data import (
    TASKS, CHANNELS, FEATURES, participant_split,
    build_source_tables, normalise_train_only, save_protocol,
)

# ----------------------------- 5. Output Folder ------------------------------
output_dir = root / "eye_movement_modelling" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100, figure_size=(9, 4))
print("Study 2 current output directory:", output_dir)
print("Shared original GazeBase source:", root / "data" / "raw" / "gazebase")


**02. Select the Original GazeBase Study Subset**

Choose complete participants with all four original task recordings in GazeBase Round 1 / Session 1. **Only MAX_PARTICIPANTS is intended to change for the 10 → 20/30-person rerun.** The source directory already contains the ten-person subset used in Study 1; a 20- or 30-person study requires additional authorised original source CSVs. The pipeline will stop rather than silently treat an incomplete 20-person request as a full cohort.

The selected participant set is ordered by original source ID, not randomly sampled. Any main-evaluation interpretation must acknowledge this selection mechanism.


In [ ]:
# ================== 02. Select Original GazeBase Recordings ==================

# ------------------------- 1. User-Facing Cohort Size -------------------------
MAX_PARTICIPANTS = 10
SEED = 17
source_dir = root / "data" / "raw" / "gazebase"

# ------------------------ 2. Discover Complete Records -----------------------
inventory = discover_records(
    source_dir, round_id=1, session_id=1,
    tasks=TASKS, max_participants=MAX_PARTICIPANTS,
)
actual_people = inventory["participant_id"].nunique()
if actual_people != MAX_PARTICIPANTS:
    raise ValueError(
        f"Requested {MAX_PARTICIPANTS} complete participants but only "
        f"{actual_people} are available. Add authorised GazeBase source CSVs "
        "before increasing the cohort size."
    )
inventory.drop(columns=["path"]).to_csv(
    output_dir / "01_source_inventory.csv", index=False,
)
print("Complete participants:", actual_people)
print("Original task recordings:", len(inventory))
print("Source task labels:", sorted(inventory.task.unique()))
display(inventory[["record_id", "participant_id", "task", "relative_file"]].head(12))


**03. Register Participant-Disjoint Train, Validation and Test**

Every selected participant contributes all four source tasks and is assigned to one partition only. The ten-person pilot uses a reproducible seeded group split; the Test portion is **exploratory**, as prior pilot decisions may influence later modelling. With at least 20 participants, original IDs beyond the initial pilot are held out for main Test, and train/validation are split among the remaining people.

Avoid interpreting a different Test assignment after changing MAX_PARTICIPANTS as an independent reproduction. Choose the intended full-study cohort size **before** inspecting main Test results.


In [ ]:
# =============== 03. Assign Independent Participant Groups =================

# -------------------------- 1. Frozen Split Registry -------------------------
split = participant_split(inventory, seed=SEED)
split.to_csv(output_dir / "01_split_assignments.csv", index=False)

# ----------------------- 2. Verify and Display Scope --------------------------
print("Study purpose:", split.study_purpose.iloc[0])
print("Independent people per partition:")
display(split.groupby("split").agg(
    n_people=("participant_id", "nunique"),
    n_task_records=("record_id", "count"),
).reset_index())
print("Participant sets:")
for role in ("train", "validation", "test"):
    people = sorted(split.loc[split.split.eq(role), "participant_id"].unique())
    print(role, ":", people)


**04. Build Study-Owned Record Features and Motion Windows**

Read each original CSV **once**, compute descriptive record features and select a bounded number of complete, contiguous valid windows. No window crosses a source-invalid sample or a timing discontinuity greater than 5 ms. Within each window, x/y are centred to their own medians and first differences are retained in physical gaze units. This is a separate preprocessing design from Study 1; its choices and results must not be substituted into Study 1.

The initial design uses **256-sample** windows and **24 evenly distributed, non-overlapping windows per recording at most**. It limits compute; it does not make extra people. Excluding records with no complete valid window requires explicit inspection, not silent row-dropping.


In [ ]:
# ================ 04. Construct Gaze Features and Windows ===================

# ----------------------- 1. Bounded Sequence Parameters ----------------------
WINDOW_SAMPLES = 256
MAX_WINDOWS_PER_RECORD = 24

# ---------------------- 2. Process Original Source Files ---------------------
record_features, windows, window_record_ids, window_labels = (
    build_source_tables(
        inventory, split, window_samples=WINDOW_SAMPLES,
        max_windows_per_record=MAX_WINDOWS_PER_RECORD,
    )
)
record_features.to_csv(output_dir / "01_record_features.csv", index=False)

# ------------------------- 3. Show Input Dimensions --------------------------
print("Source feature rows:", len(record_features))
print("Temporal windows / samples / channels:", windows.shape)
print("Raw temporal channels:", CHANNELS)
display(record_features[[
    "record_id", "participant_id", "task", "split",
    "valid_fraction", "n_windows", "speed_p95_dva_s",
]].head(12))


**05. Fit Temporal Normalisation on Training Participants Only**

Compute per-channel mean and scale using **training windows only**. Apply those fixed values to Validation and Test; do not refit after seeing either set. Tabular models also perform their own train-only imputation/scaling within scikit-learn Pipelines.

The output NPZ contains normalised windows, task indices, and source record IDs, not participant IDs as model features. Save the normalisation parameters separately for audit and future compatible prediction.


In [ ]:
# ================ 05. Train-Only Motion Normalisation ======================

# -------------------------- 1. Calculate Train Moments -----------------------
scaled, centre, scale = normalise_train_only(
    windows, window_record_ids, split,
)
np.savez_compressed(
    output_dir / "01_sequence_windows.npz",
    x=scaled, y=window_labels, record_id=window_record_ids,
)
pd.DataFrame({
    "channel": CHANNELS, "train_mean": centre, "train_scale": scale,
}).to_csv(output_dir / "01_sequence_normalisation.csv", index=False)

# ------------------------- 2. Show Train Statistics --------------------------
print("Normalised windows:", scaled.shape)
display(pd.DataFrame({
    "channel": CHANNELS, "train_mean": centre, "train_scale": scale,
}))
print("No Validation or Test examples were fitted during normalisation.")


**06. Save the Scientific Protocol and Source Overview**

Save a compact JSON configuration and a visual summary of participant groups. These files define the Study 2 cohort and parameter contract for all subsequent notebooks; they are **not** approvals, accuracy scores or historical run registries.

After this notebook executes, review the actual selected IDs, valid fraction distribution, number of windows by task, and the group partitions before training a model.


In [ ]:
# ==================== 06. Save Protocol and Source Overview ================

# ---------------------- 1. Record Explicit Scientific Setup -------------------
protocol = save_protocol(
    output_dir / "01_protocol.json",
    participants=MAX_PARTICIPANTS, seed=SEED,
    window_samples=WINDOW_SAMPLES,
    max_windows_per_record=MAX_WINDOWS_PER_RECORD,
    purpose=split.study_purpose.iloc[0],
)
display(pd.Series(protocol, name="Setting").to_frame())

# ------------------------ 2. Visualise Independent Units ---------------------
counts = split.groupby(["split", "task"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(9, 4.5))
counts.reindex(["train", "validation", "test"]).plot(
    kind="bar", ax=ax, rot=0,
)
ax.set(xlabel="Disjoint participant partition",
       ylabel="Original task recordings",
       title="GazeBase Task Availability by Participant Split")
ax.legend(title="Task")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "01_source_distribution.png")
plt.show()


**07. Scientific Interpretation and Output Download**

This initial stage establishes task labels, original record identities, participant-disjoint partitions, record-level features and train-normalised source windows. It provides **no classification performance yet**. With ten people, all downstream scores are a small-cohort pilot regardless of the number of temporal windows.

Notebook 02 consumes `01_record_features.csv`, `01_split_assignments.csv` and `01_protocol.json`. Notebooks 03/04 additionally consume `01_sequence_windows.npz`. Subsequent stages must not silently mix files from runs with different participant counts or selected records.


In [ ]:
# ===================== 07. Export Current Notebook 01 ======================

# ---------------------- 1. Confirm Current Owned Results ---------------------
print("Study 2 source records:", output_dir / "01_source_inventory.csv")
print("Participant splits:", output_dir / "01_split_assignments.csv")
print("Tabular features:", output_dir / "01_record_features.csv")
print("Temporal sequence archive:", output_dir / "01_sequence_windows.npz")
print("Protocol:", output_dir / "01_protocol.json")

# -------------------------- 2. Package Local Outputs -------------------------
download_outputs_from_colab("eye_movement_modelling/output", "01")
